# ARIA — Adaptive Reforestation Intelligence Agent
**Author:** Erneste Ntezirizaza | African Leadership University  

---

## CNN Architecture
`ARIACNNExtractor` (`env/cnn_extractor.py`) is used as the **feature extractor** inside `MultiInputPolicy`.

| Input | Shape | Processed by |
|---|---|---|
| `coverage_map` | (120,120,1) | **Spatial CNN** — detects seeded/unseeded clusters |
| `lifecycle_map` | (120,120,1) | **Spatial CNN** — detects growth stage patterns |
| `disturbance_map` | (120,120,1) | **Spatial CNN** — detects wildlife risk zones |
| `obstacle_map` | (120,120,1) | **Spatial CNN** — detects obstacle boundaries |
| `terrain_window` | (11,11,5) | **Terrain CNN** — detects local soil/slope/rain features |
| `drone_vector` | (10,) | MLP — scalar decision inputs |
| `mission_vector` | (8,) | MLP — zone quality context |
| `terrain_stats` | (6,) | MLP — zone-level generalisation features |

The CNN extracts spatial features that **transfer across zones** because terrain patterns  
(soil clusters, obstacle shapes, coverage gaps) look the same regardless of which zone the drone is in.  
A flat MLP cannot detect these patterns because it treats every cell independently after flattening.

---

## Notebook structure
| Cell | Purpose |
|---|---|
| 1 | Install dependencies |
| 2 | SB3 patch |
| 3 | Setup working directory |
| 4 | Preprocess raw GIS data |
| 5 | Build zone data |
| 6 | Reload config with real zone data |
| 7 | Environment verification |
| 8 | Training (calls `train_experiment()` from `train_ppo.py`) |
| 9 | Generate plots |
| 10 | Generalisation test |
| 11 | Save outputs |


## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'torch', 'torchvision', 'torchaudio'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch==2.4.0',
                '--index-url', 'https://download.pytorch.org/whl/cu121'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y',
                'stable-baselines3', 'shimmy', 'gymnasium'], capture_output=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'stable-baselines3==2.3.2'], check=True)

for pkg in ['rasterio', 'geopandas', 'shapely', 'pyproj',
            'xlrd', 'openpyxl', 'seaborn', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch, stable_baselines3 as sb3, gymnasium
print(f'PyTorch   : {torch.__version__}')
print(f'SB3       : {sb3.__version__}')
print(f'Gymnasium : {gymnasium.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 57.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 34.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 97.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 34.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
easyocr 1.7.2 requires torchvision>=0.5, which is not installed.
timm 1.0.26 requires torchvision, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 22.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kaggle-environments 1.29.3 requires shimmy>=1.2.1, which is not installed.
kaggle-environments 1.29.3 requires gymnasium==1.2.0, but you have gymnasium 0.29.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
2026-06-30 10:32:03.749477: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782815523.982333      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782815524.054690      23 cuda_blas.cc:1407] Unable to regist

PyTorch   : 2.4.0+cu121
SB3       : 2.3.2
Gymnasium : 0.29.1
CUDA      : True
GPU       : Tesla T4


## Cell 2 — SB3 Patch
`is_wrapped` was removed from SB3 utils in this version. Patch must run before any `PPO` import.

In [2]:
# ── SB3 compatibility patch ───────────────────────────────────
import gymnasium as _gymnasium
import stable_baselines3.common.vec_env.patch_gym as _pg
import stable_baselines3.common.base_class as _bc
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor

# Fix 1: _patch_env — accept any env-like object
def _safe_patch_env(env):
    if isinstance(env, (_gymnasium.Env, _gymnasium.Wrapper)):
        return env
    if hasattr(env, 'step') and hasattr(env, 'reset'):
        return env
    raise ValueError(f'Not a valid env: {type(env)}')

_pg._patch_env = _safe_patch_env
_bc._patch_env = _safe_patch_env

def _safe_wrap_env(env, verbose=0, monitor_wrapper=True):
    from stable_baselines3.common.vec_env import VecEnv
    from stable_baselines3.common.vec_env.base_vec_env import VecEnv as VecEnvBase
    # Duck-type check: is it already vectorised?
    is_vec = (
        isinstance(env, (VecEnv, VecEnvBase))
        or hasattr(env, 'num_envs')
    )
    if is_vec:
        return env
    # Not vectorised — add Monitor then DummyVecEnv
    if monitor_wrapper and not isinstance(env, Monitor):
        env = Monitor(env)
    env = DummyVecEnv([lambda e=env: e])
    return env

_bc.BaseAlgorithm._wrap_env = staticmethod(_safe_wrap_env)
print(' SB3 patches applied')
print('  _patch_env: accepts gymnasium envs and wrappers')
print('  _wrap_env : duck-typed VecEnv check — no double wrapping')


 SB3 patches applied
  _patch_env: accepts gymnasium envs and wrappers
  _wrap_env : duck-typed VecEnv check — no double wrapping


## Cell 3 — Setup Working Directory

In [3]:
import os, sys, shutil, importlib.util

CODE_SRC = "/kaggle/input/datasets/ernestentezirizaza2/aria-ml-code/ARIA_ML"
DATA_SRC = "/kaggle/input/datasets/ernestentezirizaza2/aria-rwanda-datasets/data"
WORK_DIR = "/kaggle/working/ARIA_ML"
OUTPUT   = "/kaggle/working/ARIA_results"

os.makedirs(OUTPUT, exist_ok=True)

# Fresh copy from input dataset
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(CODE_SRC, WORK_DIR)
print(f" Code copied to {WORK_DIR}")

# Fix ROOT_DIR so config finds data folders
with open(f"{WORK_DIR}/configs/config.py") as f:
    txt = f.read()
txt = txt.replace(
    'ROOT_DIR        = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))',
    f'ROOT_DIR        = "{WORK_DIR}"'
)
with open(f"{WORK_DIR}/configs/config.py", "w") as f:
    f.write(txt)
print(" ROOT_DIR fixed")

# Link raw datasets (symlinks save disk space)
raw_target = f"{WORK_DIR}/data/raw"
os.makedirs(raw_target, exist_ok=True)
for folder in ["dem","soil","rainfall","landcover","protected_areas","species"]:
    src = os.path.join(DATA_SRC, "raw", folder)
    dst = os.path.join(raw_target, folder)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"  Linked: {folder}")

# Create __init__.py so Python treats folders as packages
for folder in ["configs","env","utils","training"]:
    open(f"{WORK_DIR}/{folder}/__init__.py", "w").close()

# Set working directory and clean sys.path
sys.path = [p for p in sys.path if WORK_DIR not in p and "/kaggle/input" not in p]
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print(f" Ready — {WORK_DIR}")

# Helper to load modules directly from file (bypasses Kaggle sys.path issues)
def load_mod(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


 Code copied to /kaggle/working/ARIA_ML
 ROOT_DIR fixed
  Linked: dem
  Linked: soil
  Linked: rainfall
  Linked: landcover
  Linked: protected_areas
  Linked: species
 Ready — /kaggle/working/ARIA_ML


## Cell 4 — Preprocess Raw GIS Data
Converts DEM, soil, rainfall, landcover, protected areas → numpy grids.  
Elevation range derived from actual DEM. Re-run only if raw data changes.

In [4]:
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["configs","env","utils","training"]):
        del sys.modules[mod]

load_mod("configs",          f"{WORK_DIR}/configs/__init__.py")
load_mod("configs.config",   f"{WORK_DIR}/configs/config.py")
load_mod("utils",            f"{WORK_DIR}/utils/__init__.py")
load_mod("utils.preprocess", f"{WORK_DIR}/utils/preprocess.py")

from utils.preprocess import run as preprocess
preprocess()


/kaggle/working/ARIA_ML/configs/config.py:306: RuntimeWarning: Mean of empty slice
  _soil_comp    = _np.nanmean(_np.stack(_soil_layers), axis=0)


[SPECIES] Reading from: /kaggle/working/ARIA_ML/data/raw/species/Interactive Suitable Tree Species Selection and Management Tool for East Africa_Rwanda Tool.xls
[SPECIES] Bio=117r  Utils=130r x 40c
[SPECIES] Woodlot candidates: 7
  Eucalyptus globulus                 rain=500mm  (Inturusu/ ruvuvu/salinya)
  Grevillea robusta                   rain=600mm  (Gereveriya)
  Eucalyptus maculata                 rain=700mm  (Inturusu)
  Eucalyptus maidenii                 rain=800mm  (Ruvuvu)
  Eucalyptus saligna                  rain=800mm  (Salinya)
  Artocarpus heterophyllus            rain=1000mm  (Igifenesi)
  Dovyalis macrocalyx                 rain=1100mm  (Umushubi)
[SPECIES] Selected:
  sp0: Eucalyptus globulus (Inturusu/ ruvuvu/salinya) rain_min=0.0848
  sp1: Grevillea robusta (Gereveriya) rain_min=0.1018
  sp2: Eucalyptus maculata (Inturusu) rain_min=0.1188
  sp3: Eucalyptus maidenii (Ruvuvu) rain_min=0.1358
  sp4: Artocarpus heterophyllus (Igifenesi) rain_min=0.1697
ARIA — Preproce

## Cell 5 — Build Zone Data
Slices full Rwanda grid into 42 zones (36 train, 6 eval).  
Curriculum order derived from zone soil+rain scores in zone_registry.json.

In [5]:
load_mod("utils.zone_builder", f"{WORK_DIR}/utils/zone_builder.py")
from utils.zone_builder import run as build_zones
build_zones()


ARIA — Zone Builder
  Z19 [train] Northwest Highlands              soil=0.25 dist=0.01
  Z04 [train] Congo-Nile Divide North          soil=0.36 dist=0.01
  Z20 [train] North Central                    soil=0.36 dist=0.01
  Z05 [train] Congo-Nile Divide Central        soil=0.35 dist=0.01
  Z06 [eval ] Congo-Nile Divide South          soil=0.36 dist=0.01
  Z21 [train] Northeast Highlands West         soil=0.36 dist=0.01
  Z22 [train] Northeast Highlands East         soil=0.35 dist=0.01
  Z01 [train] Virunga Foothills North          soil=0.36 dist=0.01
  Z23 [train] Virunga Foothills South          soil=0.53 dist=0.01
  Z24 [train] North Central East               soil=0.52 dist=0.01
  Z10 [train] Kivu Belt North                  soil=0.51 dist=0.01
  Z12 [eval ] Kivu Belt Southwest              soil=0.53 dist=0.01
  Z25 [train] Kivu Belt Far North              soil=0.53 dist=0.01
  Z26 [train] Eastern Rift North               soil=0.52 dist=0.01
  Z27 [train] Western Ridge North         

## Cell 6 — Reload Config with Real Zone Data

`config.py` calls `_derive_from_data()` at import time to compute all thresholds from  
the actual training zone files. Since zone files didn't exist when Cell 3 ran, the  
config loaded fallback constants. This cell reloads it now that zone data exists.

**Data-derived constants:**
- `SPECIES[i]["rain_min"]` — P40/P50/P60/P75/P90 of training rainfall
- `ZONE_MIN_SOIL` — P25 of zone-level mean soil scores (aborts bottom 25% of zones)
- `ZONE_MIN_RAIN` — P25 of zone-level mean rainfall
- `RAINFALL_SUNNY_THRESH` — P50 of training rainfall (splits sunny vs rainy timesteps)
- `ELEV_MIN`, `ELEV_MAX` — actual DEM range for elevation normalisation


In [6]:
import numpy as np

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["configs","env","utils","training"]):
        del sys.modules[mod]

config = load_mod("configs.config", f"{WORK_DIR}/configs/config.py")
sys.modules["configs.config"] = config

print("=== DATA-DERIVED CONFIG THRESHOLDS ===")
print(f"TOTAL_TIMESTEPS      : {config.TOTAL_TIMESTEPS:,}")
print(f"ZONE_MIN_SOIL        : {config.ZONE_MIN_SOIL}  (P25 of zone mean soil)")
print(f"ZONE_MIN_RAIN        : {config.ZONE_MIN_RAIN}  (P25 of zone mean rain)")
print(f"RAINFALL_SUNNY_THRESH: {config.RAINFALL_SUNNY_THRESH}  (P50 of training rain)")
print(f"ELEV_MIN             : {config.ELEV_MIN}")
print(f"ELEV_MAX             : {config.ELEV_MAX}")
print()
print("Species rain_min (P40–P90 of actual training rainfall):")
for i, s in config.SPECIES.items():
    print(f"  sp{i} {s['name']:20s}: rain_min={s['rain_min']}")


/kaggle/working/ARIA_ML/configs/config.py:306: RuntimeWarning: Mean of empty slice
  _soil_comp    = _np.nanmean(_np.stack(_soil_layers), axis=0)


[SPECIES] Reading from: /kaggle/working/ARIA_ML/data/raw/species/Interactive Suitable Tree Species Selection and Management Tool for East Africa_Rwanda Tool.xls
[SPECIES] Bio=117r  Utils=130r x 40c
[SPECIES] Woodlot candidates: 7
  Eucalyptus globulus                 rain=500mm  (Inturusu/ ruvuvu/salinya)
  Grevillea robusta                   rain=600mm  (Gereveriya)
  Eucalyptus maculata                 rain=700mm  (Inturusu)
  Eucalyptus maidenii                 rain=800mm  (Ruvuvu)
  Eucalyptus saligna                  rain=800mm  (Salinya)
  Artocarpus heterophyllus            rain=1000mm  (Igifenesi)
  Dovyalis macrocalyx                 rain=1100mm  (Umushubi)
[SPECIES] Selected:
  sp0: Eucalyptus globulus (Inturusu/ ruvuvu/salinya) rain_min=0.0848
  sp1: Grevillea robusta (Gereveriya) rain_min=0.1018
  sp2: Eucalyptus maculata (Inturusu) rain_min=0.1188
  sp3: Eucalyptus maidenii (Ruvuvu) rain_min=0.1358
  sp4: Artocarpus heterophyllus (Igifenesi) rain_min=0.1697
=== DATA-DERIVE

## Cell 7 — Environment Verification

Checks:
1. Gymnasium env_checker — validates observation/action spaces
2. Zone ID fix — `zone_id=0` is falsy but valid; must use `is None` check
3. Eval zone suitability — which zones will seed vs abort


In [7]:
from gymnasium.utils.env_checker import check_env
from env.rwanda_env import RwandaReforestEnv

env = RwandaReforestEnv(zone_id=1, split="train", seed=42)
check_env(env)
env.close()
print(" Environment passed all Gymnasium checks\n")

 Environment passed all Gymnasium checks



/usr/local/lib/python3.12/dist-packages/gymnasium/utils/env_checker.py:321: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantialising the environment through gymnasium.make
  logger.warn(


## Cell 8 — Training

Calls `train_experiment(cfg)` from `train_ppo.py` for each experiment.  
All PPO decisions (policy, architecture, callbacks) live in `train_ppo.py`.  
This cell handles only orchestration: disk management, output copying, session resumption.

The CNN extractor (`env/cnn_extractor.py`) is passed via `policy_kwargs` inside  
`train_experiment()` for experiments where `use_cnn=True`.


In [8]:
import warnings, glob, json, gc, csv
import torch

warnings.filterwarnings("ignore")

from training.train_ppo import (
    PPO_EXPERIMENTS, TOTAL_TIMESTEPS,
    train_experiment, save_experiment_csv,
)
from configs.config import CHECKPOINTS_DIR

os.makedirs(OUTPUT, exist_ok=True)

# Free old checkpoint zips to save disk
for f in glob.glob(f"{CHECKPOINTS_DIR}/**/*.zip", recursive=True):
    if "best_model" not in f and "ppo_final" not in f:
        os.remove(f)

print(f"Disk space:")
os.system("df -h /kaggle/working")

# Session resumption — skip experiments already completed
completed_csv = f"{OUTPUT}/completed_experiments.csv"
completed = []
if os.path.exists(completed_csv):
    with open(completed_csv) as f:
        completed = [row["exp_id"] for row in csv.DictReader(f)]
    print(f"\nAlready completed: {completed}")

all_results = []
best_reward = -np.inf
best_dir    = None

for cfg in PPO_EXPERIMENTS:
    exp_id = cfg["exp"]

    if exp_id in completed:
        print(f"Skipping Exp {exp_id} — already done")
        result_path = f"{OUTPUT}/checkpoints/ppo_exp_{exp_id}/result.json"
        if os.path.exists(result_path):
            with open(result_path) as f:
                result = json.load(f)
            all_results.append(result)
            rew = result.get("eval_results",{}).get("mean_rewards",[])
            if rew and max(rew) > best_reward:
                best_reward = max(rew)
                best_dir    = result["checkpoint"]
        continue

    # Free disk space before each experiment
    import glob as _g
    for _f in _g.glob(f'{CHECKPOINTS_DIR}/**/*.zip', recursive=True):
        if 'best_model' not in _f and 'ppo_final' not in _f:
            os.remove(_f)
    gc.collect()
    torch.cuda.empty_cache()

    # train_experiment() handles all PPO config — notebook just calls it
    ckpt_dir = f"{OUTPUT}/checkpoints/ppo_exp_{exp_id}"
    result   = train_experiment(cfg, output_dir=ckpt_dir)
    all_results.append(result)

    rew = result.get("eval_results",{}).get("mean_rewards",[])
    if rew and max(rew) > best_reward:
        best_reward = max(rew)
        best_dir    = ckpt_dir

    # Copy best/final model to output folder
    src_dir = os.path.join(CHECKPOINTS_DIR, f"ppo_exp_{exp_id}")
    for fname in ["best_model.zip", "ppo_final.zip"]:
        src = os.path.join(src_dir, fname)
        dst = os.path.join(ckpt_dir, fname)
        if os.path.exists(src) and src != dst:
            shutil.copy2(src, dst)
            print(f"  Copied: {fname}")

    # Mark as complete for session resumption
    completed.append(exp_id)
    with open(completed_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["exp_id"])
        w.writeheader()
        for e in completed:
            w.writerow({"exp_id": e})

    gc.collect()
    torch.cuda.empty_cache()
    print("  Disk:"); os.system("df -h /kaggle/working | tail -1")

save_experiment_csv(all_results)
print(f"\n Training complete — best reward: {best_reward:.2f}  ({best_dir})")

Disk space:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.6G   17G  14% /kaggle/working

── PPO Experiment 01/3 ──
   Baseline + domain randomisation
   LR=0.0003  gamma=0.99  n_steps=4096  batch=256  clip=0.2  ent=0.1  curriculum=False
   Timesteps: 200,000
Using cuda device
---------------------------------
| eval/              |          |
|    mean_ep_length  | 333      |
|    mean_reward     | 1.01e+03 |
| time/              |          |
|    total_timesteps | 5000     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 57.9     |
|    ep_rew_mean     | 36.3     |
| time/              |          |
|    fps             | 113      |
|    iterations      | 1        |
|    time_elapsed    | 71       |
|    total_timesteps | 8192     |
---------------------------------
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 400 

## Cell 9 — Generate Plots

- **Plot 1**: Cumulative reward curves — all 3 experiments
- **Plot 2**: Entropy curves — effect of `ent_coef` on exploration
- **Plot 3**: Convergence comparison — best experiment highlighted by **peak** reward
- **Plot 4**: Generalisation test results — per eval zone metrics
- **Plot 5**: Hyperparameter summary table


In [9]:
from training.train_ppo import (
    plot_cumulative_rewards, plot_entropy_curves,
    plot_convergence, plot_hyperparameter_table,
)

plot_cumulative_rewards(all_results)
plot_entropy_curves(all_results)
plot_convergence(all_results)          # best = peak reward across all timesteps
plot_hyperparameter_table(all_results)
print(" Plots 1, 2, 3, 5 saved")


 Plot 1 saved: /kaggle/working/ARIA_ML/results/plots/plot1_cumulative_rewards.png
 Plot 2 saved: /kaggle/working/ARIA_ML/results/plots/plot2_entropy_curves.png
 Plot 3 saved: /kaggle/working/ARIA_ML/results/plots/plot3_convergence.png
 Plot 5 saved: /kaggle/working/ARIA_ML/results/plots/plot5_hyperparameter_table.png
 Plots 1, 2, 3, 5 saved


## Cell 10 — Generalisation Test

Tests the best model (highest peak reward) on all 6 held-out eval zones.  
Results saved to `metrics/generalisation.csv`.


In [10]:
from training.train_ppo import generalisation_test, plot_generalisation

if best_dir:
    print(f"Best model  : {best_dir}")
    print(f"Peak reward : {best_reward:.2f}")
    gen_results = generalisation_test(best_dir)
    plot_generalisation(gen_results)   # Plot 4 saved
else:
    print("No best checkpoint found — run training first")


Best model  : /kaggle/working/ARIA_results/checkpoints/ppo_exp_02
Peak reward : 1164.79

Running generalisation test on 6 held-out eval zones...
  Found 6 eval zones (indices 0 to 5)
  Testing zone 0 (Congo-Nile Divide South)... pct_suitable_seeded = 0.009
  Testing zone 1 (Kivu Belt Southwest)... pct_suitable_seeded = 0.014
  Testing zone 2 (Central Plateau East)... pct_suitable_seeded = 0.015
  Testing zone 3 (Northern Plateau East)... pct_suitable_seeded = 0.011
  Testing zone 4 (Eastern Savanna South)... pct_suitable_seeded = 0.011
  Testing zone 5 (Southern Valley East)... pct_suitable_seeded = 0.011
  Results saved to /kaggle/working/ARIA_ML/results/metrics/generalisation.csv
 Plot 4 saved: /kaggle/working/ARIA_ML/results/plots/plot4_generalisation.png


## Cell 11 — Save All Outputs
Copies plots and metrics to `/kaggle/working/ARIA_results/` for download.

In [11]:
from configs.config import PLOTS_DIR, METRICS_DIR

for folder in [PLOTS_DIR, METRICS_DIR]:
    if os.path.exists(folder):
        dst = os.path.join(OUTPUT, os.path.basename(folder))
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        print(f"  Copied {os.path.basename(folder)}/")

print(f"\n{'='*60}")
print("ALL COMPLETE")
print(f"  Experiments  : {len(all_results)}")
print(f"  Best reward  : {best_reward:.2f}")
print(f"  Best model   : {best_dir}")
print(f"  Plots        : {OUTPUT}/plots/  (5 PNG files)")
print(f"  Metrics      : {OUTPUT}/metrics/  (CSV files)")
print(f"  Checkpoints  : {OUTPUT}/checkpoints/")
print(f"{'='*60}")
print("Download from: Output tab → ARIA_results/")


  Copied plots/
  Copied metrics/

ALL COMPLETE
  Experiments  : 3
  Best reward  : 1164.79
  Best model   : /kaggle/working/ARIA_results/checkpoints/ppo_exp_02
  Plots        : /kaggle/working/ARIA_results/plots/  (5 PNG files)
  Metrics      : /kaggle/working/ARIA_results/metrics/  (CSV files)
  Checkpoints  : /kaggle/working/ARIA_results/checkpoints/
Download from: Output tab → ARIA_results/
